In [95]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets
import requests
from datasets import Dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text_data = requests.get(url).text

In [96]:
import torch
torch.cuda.empty_cache()

In [97]:
chunk_size = 1000
chunks = [text_data[i : i + chunk_size] for i in range(0, len(text_data), chunk_size)]
dataset = Dataset.from_dict({"text": chunks})
dataset

Dataset({
    features: ['text'],
    num_rows: 1116
})

In [98]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
def decoder(input_ids):
    return tokenizer.decode(input_ids, skip_special_tokens=True)
def encoder(text):
    return tokenizer.encode(text, add_special_tokens=False)
def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    # x = input_ids, y = labels
    tokenized["labels"] = [list(ids) for ids in tokenized["input_ids"]]
    return tokenized


In [99]:
from numpy import block


tokenized_dataset = dataset.map(preprocess_function, batched=True)
blocksize = 128
x = torch.tensor(tokenized_dataset[0:(int(0.9 * len(tokenized_dataset)))]["input_ids"]).to("cuda")
y = torch.tensor(tokenized_dataset[0:(int(0.9 * len(tokenized_dataset)))]["labels"]).to("cuda")

print(f"Success! Dataset created from raw source.")
print(f"X shape: {x.shape} | Y shape: {y.shape}")
print(f"\nFirst line of X (decoded):\n{tokenizer.decode(x[0][:60])}...")

Map:   0%|          | 0/1116 [00:00<?, ? examples/s]

Success! Dataset created from raw source.
X shape: torch.Size([1004, 128]) | Y shape: torch.Size([1004, 128])

First line of X (decoded):
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First,...


In [100]:
block_size = 128
batch_size = 32
def get_batch(split):
    data = tokenized_dataset if split == "train" else tokenized_dataset
    ix = torch.randint(0, len(data), (batch_size,))
    full_seqs = torch.stack([torch.tensor(data[i.item()]["input_ids"]) for i in ix]) # (B, T)
    
    # X is the sequence from start to second-to-last
    x = full_seqs[:, :-1] 
    # Y is the sequence from second to the end (the "next" characters)
    y = full_seqs[:, 1:] 
    
    return x.to("cuda"), y.to("cuda")

In [101]:

class head(nn.Module):
    def __init__(self,d_k,n_embd):
        super().__init__()
        self.k=nn.Linear(n_embd,d_k,bias=False)
        self.q=nn.Linear(n_embd,d_k,bias=False)
        self.v=nn.Linear(n_embd,d_k,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k = self.k(x)
        q = self.q(x)
        v = self.v(x)
        # 1. Compute scores and scale by sqrt of head_size (d_k)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5) 
        # 2. Mask future tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        # 3. Apply softmax to the normalized scores
        wei = F.softmax(wei, dim=-1) 
        out = wei @ v
        return out
        return out
    
          
        

In [102]:
dropout_rate = 0.1
class MultiheadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([head(head_size, n_embd) for _ in range(num_heads)])
        # This projection layer ensures the shape is exactly n_embd (128)
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout= nn.Dropout(dropout_rate)
    def forward(self, x):
        # 1. Run all heads: each returns (B, T, 32)
        # 2. Concatenate: results in (B, T, 32*4) = (B, T, 128)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        
        # 3. Project back to n_embd
        out = self.dropout(self.proj(out))
        return out

In [103]:

class feedforward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(n_embd,4 * n_embd),
            nn.ReLU(),
            nn.Linear(4* n_embd,n_embd),
            nn.Dropout(dropout_rate)
        )
    def forward(self,x):
        return self.net(x)    

In [104]:
class block(nn.Module):
    def __init__(self,n_embd,n_head):
        super().__init__()
        head_size=n_embd//n_head
        self.sa=MultiheadAttention(4, n_embd//4,n_embd)
        self.ffnd=feedforward(n_embd)
        self.ln1=nn.LayerNorm(n_embd)
        self.ln2=nn.LayerNorm(n_embd)   
    def forward(self,x):
        x= x+self.sa(self.ln1(x))
        x=self.ffnd(self.ln2(x)) +x
        return x

In [105]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd=128): 
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Position embedding needs to be size of context (block_size)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # The LM Head: Projects hidden states (n_embd) back to vocab_size
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.ln_f=nn.LayerNorm(n_embd)
        self.block=nn.Sequential(
            block(n_embd,n_head=4),
            block(n_embd,n_head=4),
            block(n_embd,n_head=4),
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        positions = torch.arange(T, device=idx.device) 
        pos_emb = self.position_embedding_table(positions) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)
        x = self.block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]          
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
m=BigramLanguageModel(tokenizer.vocab_size).to("cuda")    
idx=torch.zeros((1,1),dtype=torch.long).to("cuda")
    
    

In [106]:
optimizer=torch.optim.Adam(m.parameters(),lr=2e-3)
batch_size=32
for epoch in range(5000):
    xb,yb=get_batch("train")
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 10.971638679504395
Epoch 100, Loss: 5.532031536102295
Epoch 200, Loss: 4.453612327575684
Epoch 300, Loss: 3.548016309738159
Epoch 400, Loss: 2.9821579456329346
Epoch 500, Loss: 2.30236554145813
Epoch 600, Loss: 2.056568145751953
Epoch 700, Loss: 1.9432255029678345
Epoch 800, Loss: 1.4757165908813477
Epoch 900, Loss: 1.3190839290618896
Epoch 1000, Loss: 1.1495981216430664
Epoch 1100, Loss: 1.062725305557251
Epoch 1200, Loss: 0.9243276715278625
Epoch 1300, Loss: 0.839404821395874
Epoch 1400, Loss: 0.6888751983642578
Epoch 1500, Loss: 0.6687902212142944
Epoch 1600, Loss: 0.5022140145301819
Epoch 1700, Loss: 0.4532679617404938
Epoch 1800, Loss: 0.4278718829154968
Epoch 1900, Loss: 0.45415976643562317
Epoch 2000, Loss: 0.3875615894794464
Epoch 2100, Loss: 0.3884789049625397
Epoch 2200, Loss: 0.31016266345977783
Epoch 2300, Loss: 0.30858179926872253
Epoch 2400, Loss: 0.3698574900627136
Epoch 2500, Loss: 0.344101220369339
Epoch 2600, Loss: 0.28932639956474304
Epoch 2700, Loss: 

In [108]:
idx=torch.zeros((1,1),dtype=torch.long).to("cuda")
print(decoder(m.generate(idx,max_new_tokens=1000)[0].tolist()))

! ay, smooth, amen!
You love and see
I go, rather touch, 'tis a traitor then to fear,
Partdraw sign, which shouldOLYCUS:
Let us by war upon my wedding-an the house both, who hath held
te clamour fathom above water, and Romeo,
Like to meditation, and the prince: they will were
Upon his mother, who see what death, the prince's
now and the Duke of wail withal
Go on thee,
And poor Bolingbroke.

CORIOLANUS:
You seem'd out of HENRY VI:
I mayst:
And, artsey, I know it you, than ithip, you
I know you then I know you. I canst:
To think it you have you may it you, into you go on you

With you
To make you buy and you, I know you conscience unto him, you.
PAULI think of you, you, my guest, than you
not you
To think it you must
To think it you.
And it on you.
You have you.
I hate you, you you
You scarceAR LAURENCE: I have our hands:

ISABELLA:
MENENI had you it.
First Citizen:

First Citizen:

First Citizen:
You are yous you and you you, think it you have no remedy Citizen:
You are you you, think y